# ForestGuard — 02 Limpieza y transformación
Revisión de calidad, tratamiento de vacíos, tipos de datos, duplicados y valores atípicos.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
archivo_raw = Path("data/raw/incendios_forestales_2025_raw.csv")
if not archivo_raw.exists():
    raise FileNotFoundError("Ejecuta primero 01_extraccion_datos.ipynb.")
df = pd.read_csv(archivo_raw)
df_original = df.copy()
print("Dataset cargado:", df.shape)


## 2. Revisión inicial del dataset

Aquí hacemos una primera revisión para conocer el dataset. Observamos las primeras filas, cuántos registros y columnas tiene, cuáles son sus variables y qué tipo de dato tiene cada una.


In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

### Interpretación

El dataset contiene **435 registros y 78 variables**. Se observan variables numéricas, de texto y fechas. Algunas columnas están relacionadas directamente con la ubicación y afectación del incendio, mientras que otras corresponden a códigos o información administrativa.


## 3. Estadísticas descriptivas

In [ ]:
variables_numericas = [
    "latitud", "longitud", "mes", "hadecobert",
    "personasaf", "familiasaf", "animalesaf", "animalesmu"
]

variables_numericas = [c for c in variables_numericas if c in df.columns]
df[variables_numericas].describe().T


### Interpretación

La variable `hadecobert`, que representa las hectáreas de cobertura vegetal afectada, tiene una media aproximada de **2,90 hectáreas**, una mediana de **0,15 hectáreas** y un valor máximo de **105 hectáreas**. Esto muestra que la mayoría de los incendios tuvieron una afectación pequeña, aunque existen algunos casos con valores mucho mayores.


## 4. Valores faltantes

In [ ]:
faltantes_iniciales = pd.DataFrame({
    "Valores faltantes": df.isnull().sum(),
    "Porcentaje": (df.isnull().mean() * 100).round(2)
})

faltantes_iniciales[faltantes_iniciales["Valores faltantes"] > 0]


En esta primera revisión no aparecen valores `NaN`. Sin embargo, algunos campos de texto pueden estar vacíos y guardados como cadenas `""`, por lo que también es necesario revisar esos casos.


### 4.1 Revisión de campos vacíos en variables de texto

In [ ]:
vacios = {}

for columna in df.select_dtypes(include="object").columns:
    cantidad = (
        df[columna]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    if cantidad > 0:
        vacios[columna] = cantidad

campos_vacios = pd.Series(vacios).sort_values(ascending=False)
campos_vacios


### Interpretación

La revisión muestra que sí existen campos vacíos en varias variables de texto. Estos valores no eran reconocidos inicialmente como `NaN`, por lo que durante la limpieza se convertirán a datos faltantes reales para poder identificarlos correctamente.


## 5. Registros duplicados

In [ ]:
duplicados = df.duplicated().sum()
print("Cantidad de registros duplicados:", duplicados)


### Interpretación

No se encontraron registros completamente duplicados en el dataset, por lo que no es necesario eliminar filas por este motivo.


## 6. Revisión de variables categóricas

In [ ]:
columnas_categoricas = [
    "provincia", "canton", "parroquia", "causa",
    "niveldelev", "nivelinf", "region", "evento"
]

for columna in columnas_categoricas:
    if columna in df.columns:
        print(f"\n--- {columna.upper()} ---")
        print(df[columna].value_counts(dropna=False).head(20))


### Interpretación

La variable `causa` presenta una limitación importante, ya que **426 de los 435 registros** aparecen como “Desconocida”. Por esta razón, no se considera una variable fuerte para una futura etapa de Machine Learning.

También se observa que `nivelinf` contiene tres categorías: **Nivel 1, Nivel 2 y Nivel 3**. Esta variable podría servir como objetivo de clasificación, aunque existe un desbalance entre sus categorías.


## 9. Limpieza y preparación inicial

In [ ]:
df_limpio = df.copy()

### 9.1 Eliminación de duplicados

In [ ]:
df_limpio = df_limpio.drop_duplicates().copy()

### 9.2 Limpieza de textos y reconocimiento de valores faltantes

In [ ]:
columnas_texto = df_limpio.select_dtypes(include="object").columns

for columna in columnas_texto:
    df_limpio[columna] = df_limpio[columna].apply(
        lambda valor: valor.strip() if isinstance(valor, str) else valor
    )
    df_limpio[columna] = df_limpio[columna].replace("", np.nan)


### 9.3 Conversión de variables numéricas

In [ ]:
columnas_numericas = [
    "latitud", "longitud", "mes", "hadecobert",
    "personasaf", "familiasaf", "animalesaf", "animalesmu",
    "año", "dia", "feriado"
]

for columna in columnas_numericas:
    if columna in df_limpio.columns:
        df_limpio[columna] = pd.to_numeric(df_limpio[columna], errors="coerce")


### 9.4 Conversión de fechas

In [ ]:
columnas_fecha = ["fechadelev", "fechadecie", "fechaultim"]

for columna in columnas_fecha:
    if columna in df_limpio.columns:
        df_limpio[columna] = pd.to_datetime(
            df_limpio[columna],
            errors="coerce",
            dayfirst=True
        )


### Justificación de la limpieza

Solo se corrigieron los problemas que realmente aparecieron en los datos. Los textos vacíos se transformaron a `NaN`, se eliminaron espacios innecesarios y se corrigieron tipos numéricos y fechas. No fue necesario eliminar duplicados porque no se encontraron registros repetidos.


## 10. Revisión de valores atípicos

In [ ]:
area = df_limpio["hadecobert"].dropna()

q1 = area.quantile(0.25)
q3 = area.quantile(0.75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Límite superior:", limite_superior)
print("Cantidad de valores sobre el límite:", (area > limite_superior).sum())


### Interpretación

Se identificaron **49 registros** por encima del límite de **4,97 hectáreas**. Estos valores no se eliminan, porque pueden representar incendios reales de mayor magnitud y no necesariamente errores.


## 11. Comparación antes y después de la limpieza

In [ ]:
comparacion = pd.DataFrame({
    "Indicador": [
        "Registros",
        "Columnas",
        "Duplicados",
        "Valores faltantes detectados"
    ],
    "Antes": [
        df_original.shape[0],
        df_original.shape[1],
        df_original.duplicated().sum(),
        int(df_original.isnull().sum().sum())
    ],
    "Después": [
        df_limpio.shape[0],
        df_limpio.shape[1],
        df_limpio.duplicated().sum(),
        int(df_limpio.isnull().sum().sum())
    ]
})

comparacion


### Interpretación

El número de registros se mantiene en 435 y no se eliminaron filas por duplicados.

Los valores faltantes detectados aumentan después de la limpieza porque varios campos vacíos estaban guardados como texto y no eran reconocidos inicialmente como `NaN`. Después de reemplazarlos correctamente, se identifican **3095 valores faltantes**. Esto no representa una pérdida de información, sino una mejor identificación de los datos que ya estaban vacíos.


## 12. Preparación preliminar para Machine Learning

### Variable objetivo candidata

Se propone `nivelinf` como variable objetivo candidata porque representa el nivel de afectación del incendio y contiene tres categorías.

Sin embargo, existe un desbalance entre las clases, por lo que antes de entrenar un modelo será necesario considerar esta situación.


In [ ]:
variable_objetivo_candidata = "nivelinf"
df_limpio[variable_objetivo_candidata].value_counts(dropna=False)


### Variables de entrada candidatas

In [ ]:
X_candidatas = [
    "provincia",
    "canton",
    "mes",
    "region",
    "latitud",
    "longitud",
    "feriado"
]

X_candidatas = [c for c in X_candidatas if c in df_limpio.columns]
X_candidatas


Estas variables se consideran inicialmente porque aportan información sobre la ubicación y el momento en que se registró el incendio.

`causa` no se incluye entre las principales variables X porque 426 de los 435 registros aparecen como “Desconocida”, por lo que presenta muy poca información útil.


### Variables que no se utilizarían inicialmente

In [ ]:
variables_excluir = [
    "fid",
    "secuencian",
    "secuenciap",
    "codprov",
    "codcant",
    "codparr",
    "causa",
    "hadecobert",
    "niveldelev",
    "evento",
    "descripcio",
    "fuentesdei",
    "numerodese",
    "reponsable"
]

variables_excluir = [c for c in variables_excluir if c in df_limpio.columns]
variables_excluir


### Justificación de exclusiones

Se dejan fuera identificadores y códigos administrativos porque no representan características útiles del incendio.

También se excluye inicialmente `causa` por su falta de variedad. `niveldelev` y `evento` tampoco aportan diferenciación si presentan el mismo valor en todos o casi todos los registros.

`hadecobert` se mantiene fuera de las entradas hasta comprobar si está directamente relacionada con la forma en que se define `nivelinf`. Si existe esa relación, utilizarla podría provocar fuga de información.


## 13. Exportar el dataset limpio

In [ ]:
from pathlib import Path
ruta_processed = Path("data/processed")
ruta_processed.mkdir(parents=True, exist_ok=True)
archivo_processed = ruta_processed / "incendios_forestales_2025_limpio.csv"
df_limpio.to_csv(archivo_processed, index=False, encoding="utf-8-sig")
print(f"Dataset limpio guardado en: {archivo_processed}")
print("Importante: revisar campos de contacto antes de publicar este CSV en un repositorio público.")


## 14. Conclusión

El dataset permite analizar los incendios forestales registrados en Ecuador y contiene información que podría utilizarse posteriormente en Machine Learning.

`nivelinf` puede considerarse como una variable objetivo para un problema de clasificación, aunque existe un desbalance importante entre sus tres niveles.

También se encontraron limitaciones, como la gran cantidad de registros con causa desconocida y varias variables administrativas o con poca variación.

Después de la limpieza, los campos vacíos quedaron correctamente identificados como valores faltantes y se corrigieron los tipos de datos necesarios. Los valores atípicos de hectáreas afectadas se conservaron porque pueden representar incendios reales.

Con estos ajustes, los datos quedan mejor preparados para continuar hacia una siguiente etapa de Machine Learning, aunque antes será necesario revisar con mayor detalle el desbalance de la variable objetivo y seleccionar definitivamente las variables de entrada.
